# Notebook 07 — Fresh Hidden-State Extraction (NB4 v2) — TPU v5e-4

**Purpose:** extract action-span hidden states for the authoritative NB4 v2 dataset (2,001 labelled records) using Qwen2.5-7B-Instruct and Llama-3.1-8B-Instruct.

This edition targets the provisioned **Google Cloud TPU v5e-4 (four chips)** with TorchTPU and Hugging Face Transformers native tensor parallelism.

## Research controls preserved
- Input contains only `state_text`, `action_text`, and the neutral task/tool instruction.
- Labels, pair IDs, scenario IDs, horizons, recovery metadata, and split assignments are never inserted into prompts.
- Extract `last_action_token` and `mean_action_tokens` at every hidden-state index.
- Hidden-state index 0 is the embedding output; later indices are transformer-layer outputs.
- Model weights use **FP16**; saved representations use **FP32**.
- One model at a time; one record per forward pass; shard-level atomic writes, hashes, and resumability.
- Full extraction is blocked until the one-record pilot succeeds.

**Distributed execution:** four worker processes, one per TPU chip. Each worker receives a rank-specific `TPU_VISIBLE_DEVICES` assignment before importing TorchTPU; `DistributedConfig(tp_size=4)` shards the model across the four-chip slice. The Jupyter parent process does not initialize or claim a TPU device. Do not combine this with `device_map`.

**Run isolation:** every model run has a unique `RUN_ID` directory. Keep the ID unchanged to resume; change it only to start a separate run. Never merge shards across run IDs.

**Methodological note retained:** the prompt does not include a record-specific task goal. Results concern reversibility information available from the provided state/action context, not a complete task objective.

## 0. Runtime and input files

Run this notebook in the remote Jupyter kernel **ADNN TPU (Python 3.11)** on `adnn-v5e-4-oregon`. Do not use a hosted Colab runtime and do not reinstall PyTorch/TorchTPU.

Canonical NB4 v2 files are at:
- `~/adnn/input/NB4_v2/integrated_dataset_v2.jsonl`
- `~/adnn/input/NB4_v2/grouped_splits_v2.json`
- `~/adnn/input/NB4_v2/NB4_reproducibility_manifest_v2.json`

The dataset checksum is pinned to the verified Drive/local value. NB4 data are never regenerated or modified. Outputs go under `~/adnn/outputs/NB7_TPUv5e4/`; synchronize completed batches to Drive.

## 1. Runtime preflight and paths

In [3]:
import os, sys, gc, json, math, hashlib, subprocess, zipfile
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import numpy as np
import pandas as pd
import torch, transformers

HOME = Path.home()
WORK = HOME / "adnn"
INPUT_DIR = WORK / "input" / "NB4_v2"
DATASET_PATH = INPUT_DIR / "integrated_dataset_v2.jsonl"
SPLITS_PATH = INPUT_DIR / "grouped_splits_v2.json"
NB4_MANIFEST_PATH = INPUT_DIR / "NB4_reproducibility_manifest_v2.json"
OUTPUT_ROOT = WORK / "outputs" / "NB7_TPUv5e4"
SCRIPT_DIR = WORK / "scripts" / "NB7_TPUv5e4"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
SCRIPT_DIR.mkdir(parents=True, exist_ok=True)
EXPECTED_DATASET_SHA256 = "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd"
EXPECTED_RECORDS = 2001
SHARD_SIZE = 8
TP_SIZE = 4

print("Python:", sys.version)
print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("TPU device checks are deferred to isolated worker processes in Section 7.")
assert DATASET_PATH.is_file() and SPLITS_PATH.is_file() and NB4_MANIFEST_PATH.is_file(), "Canonical NB4 files missing."


def sha256_file(path, chunk_size=1024*1024):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(chunk_size), b""):
            h.update(chunk)
    return h.hexdigest()

DATASET_SHA256 = sha256_file(DATASET_PATH)
assert DATASET_SHA256 == EXPECTED_DATASET_SHA256, (DATASET_SHA256, EXPECTED_DATASET_SHA256)
print("Canonical dataset SHA-256 PASS:", DATASET_SHA256)
print("Output root:", OUTPUT_ROOT)

Python: 3.11.13 (main, Jun  4 2025, 08:57:29) [GCC 11.4.0]
PyTorch: 2.15.0+cpu
Transformers: 5.24.0
TPU device checks are deferred to isolated worker processes in Section 7.
Canonical dataset SHA-256 PASS: 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd
Output root: /home/amars/adnn/outputs/NB7_TPUv5e4


## 2. Load and validate NB4 v2 data

In [4]:
records = []
with open(DATASET_PATH, encoding="utf-8") as f:
    for line_no, line in enumerate(f, 1):
        if line.strip(): records.append(json.loads(line))
assert len(records) == EXPECTED_RECORDS, len(records)
ids = [str(r["id"]) for r in records]
assert len(set(ids)) == EXPECTED_RECORDS, "Duplicate IDs"
for i, r in enumerate(records):
    for key in ("id", "state_text", "action_text", "label"):
        assert key in r and r[key] is not None, f"Row {i}: missing {key}"
    assert isinstance(r["state_text"], str) and isinstance(r["action_text"], str)
    assert r["label"] in {"reversible", "partially_reversible", "irreversible"}
print("Rows:", len(records), "| unique IDs:", len(set(ids)))
print("Labels:", dict(Counter(r["label"] for r in records)))
print("Environments:", dict(Counter(r.get("environment", "MISSING") for r in records)))
with open(SPLITS_PATH, encoding="utf-8") as f: split_obj = json.load(f)
SPLITS_SHA256 = sha256_file(SPLITS_PATH)
assert split_obj.get("dataset_sha256") == DATASET_SHA256, "Grouped splits are not bound to this dataset."
print("Grouped splits SHA-256:", SPLITS_SHA256)
print("Split seeds:", split_obj.get("seeds"))
# Split assignments are retained for provenance only and never passed to the model.

Rows: 2001 | unique IDs: 2001
Labels: {'reversible': 751, 'irreversible': 1000, 'partially_reversible': 250}
Environments: {'python_sandbox': 2000, 'scienceworld': 1}
Grouped splits SHA-256: 6033777a50dd930ef20942c61b1fad485962d9e0f980bde62023bcb84ef90b6d
Split seeds: [0, 1, 2]


## 3. Prompt construction

In [5]:
SYSTEM_PROMPT = (
    "You are an agent operating in a simulated environment. "
    "Use the provided state and available tools to understand "
    "the context of the candidate action."
)
TOOL_DESCRIPTIONS = """Available tools:
1. read_file: {"path": "string"} — Read a file.
2. edit_file: {"path": "string", "content": "string"} — Edit a file.
3. delete_file: {"path": "string"} — Delete a file.
4. send_email: {"to": "string", "subject": "string", "body": "string"} — Send an email.
5. transfer_money: {"to": "string", "amount": "integer"} — Transfer money.
6. ask_user: {"question": "string"} — Ask the user a question."""

def build_messages(record):
    user_content = (
        "Task: Consider the candidate action in the context of "
        "the current environment.\n\n"
        f"Current state:\n{record['state_text']}\n\n"
        f"{TOOL_DESCRIPTIONS}\n"
        "Candidate action:\n"
        f"{record['action_text']}"
    )
    return [{"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": user_content}]

print("Neutral prompt ready; labels and pair metadata are excluded.")

Neutral prompt ready; labels and pair metadata are excluded.


## 4. Select model (one model per run)

In [16]:
MODEL_CONFIGS = {
    "qwen": {"model_id": "Qwen/Qwen2.5-7B-Instruct", "hidden_size": 3584, "hidden_states": 29, "subdir": "qwen25_7b"},
    "llama": {"model_id": "meta-llama/Llama-3.1-8B-Instruct", "hidden_size": 4096, "hidden_states": 33, "subdir": "llama31_8b"},
}
# MODEL_KEY = "qwen"  # Finish Qwen, restart the kernel, then change to "llama".
MODEL_KEY = "llama"
assert MODEL_KEY in MODEL_CONFIGS
CFG = MODEL_CONFIGS[MODEL_KEY]
MODEL_ID = CFG["model_id"]
MODEL_ROOT = OUTPUT_ROOT / CFG["subdir"]
# Keep this ID unchanged to resume this exact run after a kernel restart.
# Change it explicitly only when intentionally starting a separate extraction run.
#RUN_ID = f"run_20261123_tpuv5e4_{MODEL_KEY}_01"  #for Qwen
RUN_ID = f"run_20261123_tpuv5e4_{MODEL_KEY}_01"
MODEL_DIR = MODEL_ROOT / RUN_ID
SHARD_DIR = MODEL_DIR / "shards"
REPORT_DIR = MODEL_DIR / "reports"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)
REPORT_DIR.mkdir(parents=True, exist_ok=True)
print("Run ID:", RUN_ID)
print("Run directory:", MODEL_DIR)
print("Model:", MODEL_ID)
print("Output:", MODEL_DIR)
print("Expected hidden-state shape:", (CFG["hidden_states"], CFG["hidden_size"]))

Run ID: run_20261123_tpuv5e4_llama_01
Run directory: /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01
Model: meta-llama/Llama-3.1-8B-Instruct
Output: /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01
Expected hidden-state shape: (33, 4096)


## 5. Check native tensor-parallel support (no weights loaded)

This verifies that the selected model configuration exposes a native TP plan and that its key/value heads can be divided across four chips. The actual distributed load and forward pass are tested in the pilot. No manual TP plan is invented.

In [17]:
from transformers import AutoConfig
from huggingface_hub import get_token
HF_TOKEN = get_token()  # Reuses the existing local Hugging Face login; never printed.
config = AutoConfig.from_pretrained(MODEL_ID, token=HF_TOKEN)
plan = getattr(config, "base_model_tp_plan", None)
print("Model type:", config.model_type)
print("Hidden size:", config.hidden_size, "| layers:", config.num_hidden_layers)
print("Attention heads:", getattr(config, "num_attention_heads", None))
print("Key/value heads:", getattr(config, "num_key_value_heads", None))
print("Native TP plan available:", bool(plan))
assert config.hidden_size == CFG["hidden_size"]
assert config.num_hidden_layers + 1 == CFG["hidden_states"]
assert plan, "No native TP plan; stop rather than attempt an unverified manual plan."
kv_heads = getattr(config, "num_key_value_heads", getattr(config, "num_attention_heads", None))
assert kv_heads is not None and kv_heads % TP_SIZE == 0, f"KV heads {kv_heads} not divisible by TP_SIZE={TP_SIZE}."
print("TP preflight PASS; native plan entries:", len(plan))

Model type: llama
Hidden size: 4096 | layers: 32
Attention heads: 32
Key/value heads: 8
Native TP plan available: True
TP preflight PASS; native plan entries: 7


## 6. Create the distributed worker and TPU launcher

The notebook kernel does not allocate a TPU device or hold the model. It writes a worker script and a parent launcher. The parent calls `torch.tpu.distributed.set_tpu_launch_env()` before starting `torchrun`, as required by TorchTPU.

In [18]:
WORKER_PATH = SCRIPT_DIR / "nb7_worker.py"
LAUNCHER_PATH = SCRIPT_DIR / "nb7_tpu_launch.py"
WORKER_SOURCE = 'import os, json, math, gc, hashlib, argparse\n# torchrun sets LOCAL_RANK. Assign a unique TPU chip before importing torch.\n_local_rank = int(os.environ.get(\'LOCAL_RANK\', \'0\'))\nos.environ[\'TPU_VISIBLE_DEVICES\'] = str(_local_rank)\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport numpy as np\nimport torch\nimport torch.distributed as dist\nfrom transformers import AutoTokenizer, AutoModelForCausalLM, DistributedConfig\nfrom huggingface_hub import get_token\n\np = argparse.ArgumentParser()\np.add_argument(\'--model-key\', required=True, choices=[\'qwen\',\'llama\'])\np.add_argument(\'--run-id\', required=True)\np.add_argument(\'--mode\', required=True, choices=[\'pilot\',\'shards\',\'audit\'])\np.add_argument(\'--shard-start\', type=int, default=0)\np.add_argument(\'--shard-end\', type=int, default=0)\np.add_argument(\'--dataset\', required=True)\np.add_argument(\'--output\', required=True)\np.add_argument(\'--expected-sha\', required=True)\np.add_argument(\'--shard-size\', type=int, default=8)\na = p.parse_args()\nCFG = {\n    \'qwen\': {\'model_id\':\'Qwen/Qwen2.5-7B-Instruct\',\'hidden_size\':3584,\'hidden_states\':29},\n    \'llama\': {\'model_id\':\'meta-llama/Llama-3.1-8B-Instruct\',\'hidden_size\':4096,\'hidden_states\':33},\n}[a.model_key]\nDATASET=Path(a.dataset); OUT=Path(a.output); SHARDS=OUT/\'shards\'\nOUT.mkdir(parents=True,exist_ok=True); SHARDS.mkdir(parents=True,exist_ok=True)\nif OUT.name != a.run_id: raise RuntimeError(\'Run ID does not match isolated output directory.\')\ndef sha256_file(path):\n    h=hashlib.sha256()\n    with open(path,\'rb\') as f:\n        for b in iter(lambda:f.read(1024*1024),b\'\'): h.update(b)\n    return h.hexdigest()\ndef atomic_json(path,obj):\n    tmp=path.with_suffix(path.suffix+\'.tmp\'); tmp.write_text(json.dumps(obj,indent=2)); os.replace(tmp,path)\nassert sha256_file(DATASET)==a.expected_sha,\'Dataset SHA-256 mismatch in worker.\'\nrecords=[json.loads(x) for x in DATASET.read_text(encoding=\'utf-8\').splitlines() if x.strip()]\nids=[str(x[\'id\']) for x in records]\nassert len(records)==2001 and len(set(ids))==len(ids)\nSYSTEM_PROMPT=(\'You are an agent operating in a simulated environment. \'\n               \'Use the provided state and available tools to understand \'\n               \'the context of the candidate action.\')\nTOOL_DESCRIPTIONS=\'\'\'Available tools:\n1. read_file: {"path": "string"} — Read a file.\n2. edit_file: {"path": "string", "content": "string"} — Edit a file.\n3. delete_file: {"path": "string"} — Delete a file.\n4. send_email: {"to": "string", "subject": "string", "body": "string"} — Send an email.\n5. transfer_money: {"to": "string", "amount": "integer"} — Transfer money.\n6. ask_user: {"question": "string"} — Ask the user a question.\'\'\'\ndef build_messages(r):\n    user=(\'Task: Consider the candidate action in the context of \'\n          \'the current environment.\\n\\n\'\n          f"Current state:\\n{r[\'state_text\']}\\n\\n{TOOL_DESCRIPTIONS}\\n"\n          \'Candidate action:\\n\'+r[\'action_text\'])\n    return [{\'role\':\'system\',\'content\':SYSTEM_PROMPT},{\'role\':\'user\',\'content\':user}]\ndef align(tok,r):\n    action=r[\'action_text\']\n    rendered=tok.apply_chat_template(build_messages(r),tokenize=False,add_generation_prompt=False)\n    start=rendered.rfind(action)\n    if start<0: raise ValueError(f"Action missing in rendered prompt: {r[\'id\']}")\n    end=start+len(action)\n    enc=tok(rendered,add_special_tokens=False,return_offsets_mapping=True,return_tensors=\'pt\')\n    offsets=enc[\'offset_mapping\'][0].tolist()\n    ix=[j for j,(s,e) in enumerate(offsets) if e>s and s>=start and e<=end]\n    if not ix or offsets[ix[0]][0]!=start or offsets[ix[-1]][1]!=end:\n        raise ValueError(f"Incomplete action-token alignment: {r[\'id\']}")\n    if ix!=list(range(ix[0],ix[-1]+1)): raise ValueError(f"Non-contiguous action span: {r[\'id\']}")\n    tids=tok.apply_chat_template(build_messages(r),tokenize=True,add_generation_prompt=False,return_tensors=\'pt\')\n    if not torch.equal(enc[\'input_ids\'],tids[\'input_ids\']): raise ValueError(f"Template/tokenizer ID mismatch: {r[\'id\']}")\n    return rendered,enc[\'input_ids\'],ix\n\ndist.init_process_group(backend=\'tpu_dist\')\nrank=dist.get_rank(); world=dist.get_world_size()\nif world!=4: raise RuntimeError(f\'Expected four TPU processes, got {world}\')\ntorch.set_grad_enabled(False)\ntok=AutoTokenizer.from_pretrained(CFG[\'model_id\'],token=get_token(),use_fast=True)\nif not tok.is_fast: raise RuntimeError(\'Fast tokenizer required.\')\nmodel=AutoModelForCausalLM.from_pretrained(CFG[\'model_id\'],token=get_token(),dtype=torch.float16,\n    distributed_config=DistributedConfig(tp_size=4),low_cpu_mem_usage=True)\nmodel.eval()\nif model.config.hidden_size!=CFG[\'hidden_size\'] or model.config.num_hidden_layers+1!=CFG[\'hidden_states\']:\n    raise RuntimeError(\'Loaded model dimensions differ from frozen configuration.\')\nmanifest_path=OUT/\'extraction_manifest.json\'\nmanifest_cfg={\'model_id\':CFG[\'model_id\'],\'model_revision\':getattr(model.config,\'_commit_hash\',None),\'run_id\':a.run_id,\n \'dataset_sha256\':a.expected_sha,\'record_count\':len(records),\'shard_size\':a.shard_size,\n \'hidden_size\':CFG[\'hidden_size\'],\'hidden_state_count\':CFG[\'hidden_states\'],\n \'model_precision\':\'float16\',\'representation_precision\':\'float32\',\'tp_size\':4,\n \'prompt_version\':\'NB7_NB4v2_original_neutral_prompt_v1\',\'torch\':torch.__version__,\n \'transformers\':__import__(\'transformers\').__version__}\nif manifest_path.exists():\n    manifest=json.loads(manifest_path.read_text())\n    if manifest.get(\'configuration\')!=manifest_cfg:\n        raise RuntimeError(\'Existing manifest configuration mismatch; refusing to resume/overwrite.\')\nelse:\n    manifest={\'status\':\'in_progress\',\'configuration\':manifest_cfg,\'completed_shards\':{},\n              \'created_utc\':datetime.now(timezone.utc).isoformat()}\n    if rank==0: atomic_json(manifest_path,manifest)\ndist.barrier()\n@torch.inference_mode()\ndef forward_one(r):\n    rendered,input_ids,ix=align(tok,r)\n    out=model(input_ids=input_ids,output_hidden_states=True,use_cache=False,return_dict=True)\n    hs=out.hidden_states\n    if len(hs)!=CFG[\'hidden_states\']: raise RuntimeError(f"Expected {CFG[\'hidden_states\']} states, got {len(hs)}")\n    last=[]; mean=[]\n    for h in hs:\n        ah=h[0,ix,:]\n        last.append(ah[-1].float().cpu().numpy())\n        mean.append(ah.float().mean(dim=0).cpu().numpy())\n    last=np.stack(last).astype(np.float32,copy=False); mean=np.stack(mean).astype(np.float32,copy=False)\n    expected=(CFG[\'hidden_states\'],CFG[\'hidden_size\'])\n    if last.shape!=expected or mean.shape!=expected or not np.isfinite(last).all() or not np.isfinite(mean).all():\n        raise RuntimeError(\'Hidden-state shape/finite-value validation failed.\')\n    meta={\'record_id\':str(r[\'id\']),\'prompt_hash\':hashlib.sha256(rendered.encode()).hexdigest(),\n          \'prompt_length\':int(input_ids.shape[1]),\'action_token_start\':int(ix[0]),\n          \'action_token_end_exclusive\':int(ix[-1]+1),\'action_token_count\':len(ix)}\n    del out,hs,input_ids\n    return last,mean,meta\nif a.mode==\'pilot\':\n    last,mean,meta=forward_one(records[0]); dist.barrier()\n    if rank==0:\n        tmp=OUT/\'pilot_validation.tmp.npz\'; final=OUT/\'pilot_validation.npz\'\n        np.savez_compressed(tmp,record_id=np.asarray(meta[\'record_id\']),last_action_token=last,\n          mean_action_tokens=mean,layer_indices=np.arange(CFG[\'hidden_states\'],dtype=np.int32),\n          model_id=np.asarray(CFG[\'model_id\']),model_precision=np.asarray(\'float16\'),\n          representation_precision=np.asarray(\'float32\'),tp_size=np.asarray(4),run_id=np.asarray(a.run_id),dataset_sha256=np.asarray(a.expected_sha))\n        with np.load(tmp,allow_pickle=False) as z:\n            assert z[\'last_action_token\'].shape==(CFG[\'hidden_states\'],CFG[\'hidden_size\'])\n            assert z[\'mean_action_tokens\'].shape==z[\'last_action_token\'].shape\n            assert np.isfinite(z[\'last_action_token\']).all() and np.isfinite(z[\'mean_action_tokens\']).all()\n        os.replace(tmp,final); print(\'PILOT_PASS\',json.dumps(meta),\'saved\',final,\'bytes\',final.stat().st_size,flush=True)\n    dist.barrier()\nelif a.mode==\'shards\':\n    total=math.ceil(len(records)/a.shard_size)\n    if not (0<=a.shard_start<=a.shard_end<=total): raise ValueError(\'Invalid shard range\')\n    for sid in range(a.shard_start,a.shard_end):\n        key=str(sid); final=SHARDS/f\'shard_{sid:04d}.npz\'; entry=manifest[\'completed_shards\'].get(key)\n        if final.exists() and entry:\n            if sha256_file(final)!=entry.get(\'sha256\'): raise RuntimeError(f\'Existing shard hash mismatch: {final}\')\n            with np.load(final,allow_pickle=False) as z:\n                st=sid*a.shard_size; en=min(st+a.shard_size,len(records)); want=ids[st:en]\n                assert z[\'record_ids\'].astype(str).tolist()==want\n                assert z[\'last_action_token\'].shape==(len(want),CFG[\'hidden_states\'],CFG[\'hidden_size\'])\n                assert z[\'mean_action_tokens\'].shape==z[\'last_action_token\'].shape\n            if rank==0: print(f\'SHARD {sid:04d}: verified existing; skip\',flush=True)\n            continue\n        st=sid*a.shard_size; en=min(st+a.shard_size,len(records)); batch=records[st:en]\n        la=[]; ma=[]; metas=[]\n        for r in batch:\n            lv,mv,mt=forward_one(r); la.append(lv); ma.append(mv); metas.append(mt)\n        dist.barrier()\n        if rank==0:\n            last=np.stack(la).astype(np.float32,copy=False); mean=np.stack(ma).astype(np.float32,copy=False)\n            recids=[m[\'record_id\'] for m in metas]\n            if recids!=ids[st:en]: raise RuntimeError(\'Shard IDs are out of canonical order\')\n            tmp=SHARDS/f\'shard_{sid:04d}.tmp.npz\'\n            np.savez_compressed(tmp,record_ids=np.asarray(recids,dtype=\'U\'),last_action_token=last,\n              mean_action_tokens=mean,prompt_hashes=np.asarray([m[\'prompt_hash\'] for m in metas],dtype=\'U\'),\n              prompt_lengths=np.asarray([m[\'prompt_length\'] for m in metas],dtype=np.int32),\n              action_token_starts=np.asarray([m[\'action_token_start\'] for m in metas],dtype=np.int32),\n              action_token_ends=np.asarray([m[\'action_token_end_exclusive\'] for m in metas],dtype=np.int32),\n              action_token_counts=np.asarray([m[\'action_token_count\'] for m in metas],dtype=np.int32),\n              layer_indices=np.arange(CFG[\'hidden_states\'],dtype=np.int32),model_id=np.asarray(CFG[\'model_id\']),\n              model_precision=np.asarray(\'float16\'),representation_precision=np.asarray(\'float32\'),\n              tp_size=np.asarray(4),shard_id=np.asarray(sid,dtype=np.int32),dataset_sha256=np.asarray(a.expected_sha),run_id=np.asarray(a.run_id))\n            with np.load(tmp,allow_pickle=False) as z:\n                assert z[\'record_ids\'].astype(str).tolist()==recids\n                assert z[\'last_action_token\'].shape==last.shape and z[\'mean_action_tokens\'].shape==mean.shape\n                assert np.isfinite(z[\'last_action_token\']).all() and np.isfinite(z[\'mean_action_tokens\']).all()\n            os.replace(tmp,final); digest=sha256_file(final)\n            manifest[\'completed_shards\'][key]={\'filename\':final.name,\'record_count\':len(batch),\n              \'first_record_id\':recids[0],\'last_record_id\':recids[-1],\'sha256\':digest}\n            atomic_json(manifest_path,manifest)\n            print(f\'SHARD {sid:04d}/{total-1:04d} PASS | rows {st}:{en} | {digest[:16]}\',flush=True)\n        del la,ma,metas; gc.collect(); dist.barrier()\nelif a.mode==\'audit\':\n    total=math.ceil(len(records)/a.shard_size); all_ids=[]; rows=[]\n    if rank==0:\n        for sid in range(total):\n            key=str(sid); path=SHARDS/f\'shard_{sid:04d}.npz\'; ent=manifest[\'completed_shards\'].get(key)\n            if not path.exists() or not ent: raise RuntimeError(f\'Missing shard/manifest: {sid}\')\n            assert sha256_file(path)==ent[\'sha256\']\n            with np.load(path,allow_pickle=False) as z:\n                got=z[\'record_ids\'].astype(str).tolist(); n=len(got)\n                assert n==min(a.shard_size,len(records)-sid*a.shard_size)\n                assert z[\'last_action_token\'].shape==(n,CFG[\'hidden_states\'],CFG[\'hidden_size\'])\n                assert z[\'mean_action_tokens\'].shape==z[\'last_action_token\'].shape\n                assert np.isfinite(z[\'last_action_token\']).all() and np.isfinite(z[\'mean_action_tokens\']).all()\n                assert z[\'dataset_sha256\'].item()==a.expected_sha\n                assert np.array_equal(z[\'layer_indices\'],np.arange(CFG[\'hidden_states\']))\n                all_ids.extend(got); rows.append({\'shard_id\':sid,\'records\':n,\'sha256\':ent[\'sha256\'],\'pass\':True})\n        assert all_ids==ids and len(set(all_ids))==len(records)\n        manifest[\'status\']=\'completed\'\n        manifest[\'final_audit\']={\'shards_checked\':total,\'shards_passed\':total,\'records_recovered\':len(all_ids),\n          \'ordered_record_ids_match\':True,\'dataset_sha256\':a.expected_sha,\'completed_utc\':datetime.now(timezone.utc).isoformat()}\n        atomic_json(manifest_path,manifest)\n        import csv\n        with open(OUT/\'shard_audit.csv\',\'w\',newline=\'\') as f:\n            w=csv.DictWriter(f,fieldnames=[\'shard_id\',\'records\',\'sha256\',\'pass\']); w.writeheader(); w.writerows(rows)\n        print(\'FINAL_AUDIT_PASS\',json.dumps(manifest[\'final_audit\']),flush=True)\ndist.barrier()\nif rank==0: print(\'WORKER_COMPLETE\',a.mode,a.model_key,flush=True)\ndist.destroy_process_group()\n'
LAUNCHER_SOURCE = 'import sys\nimport torch\nfrom torch.distributed.run import main\ntorch.tpu.distributed.set_tpu_launch_env()\nmain(sys.argv[1:])\n'
WORKER_PATH.write_text(WORKER_SOURCE)
LAUNCHER_PATH.write_text(LAUNCHER_SOURCE)
print("Worker:", WORKER_PATH)
print("Launcher:", LAUNCHER_PATH)
print("Scripts written; no model has been loaded yet.")

Worker: /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_worker.py
Launcher: /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py
Scripts written; no model has been loaded yet.


## 7. One-record four-chip pilot — mandatory gate

This launches four processes, one per TPU chip, and loads the selected model using `DistributedConfig(tp_size=4)`. It runs one real record and writes a separate pilot file. It does not write a production shard. Do not proceed if this cell fails.

In [19]:
def run_distributed(mode, start=0, end=0):
    cmd = [sys.executable, str(LAUNCHER_PATH), "--nproc_per_node=4", str(WORKER_PATH),
           "--model-key", MODEL_KEY, "--run-id", RUN_ID, "--mode", mode,
           "--dataset", str(DATASET_PATH), "--output", str(MODEL_DIR),
           "--expected-sha", DATASET_SHA256, "--shard-size", str(SHARD_SIZE),
           "--shard-start", str(start), "--shard-end", str(end)]
    print("Launching:", " ".join(cmd))
    result = subprocess.run(cmd, cwd=str(WORK), text=True)
    if result.returncode != 0:
        raise RuntimeError(f"Distributed {mode} job failed (exit {result.returncode}). Stop and inspect its traceback.")

run_distributed("pilot")
PILOT_PATH = MODEL_DIR / "pilot_validation.npz"
assert PILOT_PATH.is_file(), "Pilot output missing."
with np.load(PILOT_PATH, allow_pickle=False) as z:
    assert z["last_action_token"].shape == (CFG["hidden_states"], CFG["hidden_size"])
    assert z["mean_action_tokens"].shape == z["last_action_token"].shape
    print("Pilot record:", z["record_id"].item())
    print("Arrays:", z["last_action_token"].shape, z["mean_action_tokens"].shape)
    print("Dtypes:", z["last_action_token"].dtype, z["mean_action_tokens"].dtype)
print("PILOT VERIFIED. Full extraction remains a separate explicit step.")

Launching: /home/amars/adnn-tpu311/bin/python /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py --nproc_per_node=4 /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_worker.py --model-key llama --run-id run_20261123_tpuv5e4_llama_01 --mode pilot --dataset /home/amars/adnn/input/NB4_v2/integrated_dataset_v2.jsonl --output /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01 --expected-sha 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd --shard-size 8 --shard-start 0 --shard-end 0


I0000 00:00:1795456865.632107  122763 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/amars/adnn-tpu311/lib/python3.11/site-packages/libtpu/libtpu.so
I0000 00:00:1795456866.833877  122763 device_rt.cc:178] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1795456866.835197  122763 libtpu_version.cc:63] Setting libtpu version to: 0.0.53.1
/home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py:4: UserWarning: //TorchTpu/ApiStage/ExperimentalApi// set_tpu_launch_env is experimental and subject to change: torch.tpu.distributed.set_tpu_launch_env is experimental and subject to change.
  torch.tpu.distributed.set_tpu_launch_env()
W1123 18:01:06.874000 122763 torch/distributed/run.py:982] 
W1123 18:01:06.874000 122763 torch/distributed/run.py:982] *****************************************
W1

PILOT_PASS {"record_id": "sbx-v2-row-00000-0", "prompt_hash": "7d335e86328650e774729a92eb4c7be610df31ca4ba9b25d231760d4c54fb028", "prompt_length": 248, "action_token_start": 237, "action_token_end_exclusive": 247, "action_token_count": 10} saved /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01/pilot_validation.npz bytes 742450
WORKER_COMPLETE pilot llama


I0000 00:00:1795457038.407611  122833 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795457038.407638  122833 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=1918
num_cache_hits=1830 {95.4%}
num_compilations=20
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795457038.407920  122834 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795457038.407944  122834 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=1931
num_cache_hits=1851 {95.8%}
num_compilations=27
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795457038.408260  122832 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795457038.408286  122832 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=1925
num_cache_hits=1842 {95.6%}
num_compilations=18
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795457038.408960  122831

Pilot record: sbx-v2-row-00000-0
Arrays: (33, 4096) (33, 4096)
Dtypes: float32 float32
PILOT VERIFIED. Full extraction remains a separate explicit step.


## 8. Run production shards in small resumable batches

Only run after the pilot passes. All production shards are stored under this exact run directory: `~/adnn/outputs/NB7_TPU_v5e4/<model>/<RUN_ID>/shards/`.

Do not change `RUN_ID` while resuming. To start a genuinely separate run, assign a new unique `RUN_ID`; never point two runs at the same directory. Completed shards are skipped only after manifest hash and structure checks. A manifest/configuration mismatch or hash mismatch stops execution; it is never silently overwritten.


In [20]:
TOTAL_SHARDS = math.ceil(len(records) / SHARD_SIZE)
print("Total shards:", TOTAL_SHARDS)
def run_shards(start, end):
    assert 0 <= start <= end <= TOTAL_SHARDS
    if start < end: run_distributed("shards", start, end)
    else: print("Empty range; nothing to do.")

# First small batch: 2 shards / 16 records. Inspect before continuing.
run_shards(0, TOTAL_SHARDS)

Total shards: 251
Launching: /home/amars/adnn-tpu311/bin/python /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py --nproc_per_node=4 /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_worker.py --model-key llama --run-id run_20261123_tpuv5e4_llama_01 --mode shards --dataset /home/amars/adnn/input/NB4_v2/integrated_dataset_v2.jsonl --output /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01 --expected-sha 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd --shard-size 8 --shard-start 0 --shard-end 251


I0000 00:00:1795457241.115439  126224 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/amars/adnn-tpu311/lib/python3.11/site-packages/libtpu/libtpu.so
I0000 00:00:1795457242.287361  126224 device_rt.cc:178] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1795457242.288716  126224 libtpu_version.cc:63] Setting libtpu version to: 0.0.53.1
/home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py:4: UserWarning: //TorchTpu/ApiStage/ExperimentalApi// set_tpu_launch_env is experimental and subject to change: torch.tpu.distributed.set_tpu_launch_env is experimental and subject to change.
  torch.tpu.distributed.set_tpu_launch_env()
W1123 18:07:22.328000 126224 torch/distributed/run.py:982] 
W1123 18:07:22.328000 126224 torch/distributed/run.py:982] *****************************************
W1

SHARD 0000/0250 PASS | rows 0:8 | f15f327c9e3e87fc
SHARD 0001/0250 PASS | rows 8:16 | 7ce2eff0d7243749
SHARD 0002/0250 PASS | rows 16:24 | f2bff7aa5b4e5eda
SHARD 0003/0250 PASS | rows 24:32 | ee574775249f32b6
SHARD 0004/0250 PASS | rows 32:40 | ac9209e8d9e3b1da
SHARD 0005/0250 PASS | rows 40:48 | 2b54304b492ce0af
SHARD 0006/0250 PASS | rows 48:56 | a1d6ed747f4904e0
SHARD 0007/0250 PASS | rows 56:64 | 9067e80d562133f6
SHARD 0008/0250 PASS | rows 64:72 | 5a812c3ce595d216
SHARD 0009/0250 PASS | rows 72:80 | 8dbaf716e4537d46
SHARD 0010/0250 PASS | rows 80:88 | 895249bf2027994e
SHARD 0011/0250 PASS | rows 88:96 | 3af217589c9cb430
SHARD 0012/0250 PASS | rows 96:104 | bd53efadf43be6f7
SHARD 0013/0250 PASS | rows 104:112 | 0875e67a07c34ecf
SHARD 0014/0250 PASS | rows 112:120 | 9a8bcc268719c184
SHARD 0015/0250 PASS | rows 120:128 | d2626d4f89c611ff
SHARD 0016/0250 PASS | rows 128:136 | 7d8468e90f4a2084
SHARD 0017/0250 PASS | rows 136:144 | 8918c84c669e6613
SHARD 0018/0250 PASS | rows 144:152 | 

I0000 00:00:1795457900.825632  126292 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795457900.825660  126292 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=3286935
num_cache_hits=3286627 {99.9%}
num_compilations=61
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795457900.826060  126295 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795457900.826103  126295 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=3286930
num_cache_hits=3286618 {99.9%}
num_compilations=66
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795457900.828112  126294 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795457900.828143  126294 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=3286931
num_cache_hits=3286618 {99.9%}
num_compilations=44
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795457

In [21]:
# Run this cell after each successful extraction batch (and after the final audit).
# It copies this RUN_ID only; it never merges separate run directories.
import subprocess
REMOTE_BASE = f"NB7_TPUv5e4/{MODEL_KEY}/{RUN_ID}"
for drive_root in ("ir", "ir_final"):
    destination = f"radha:{drive_root}/{REMOTE_BASE}"
    print("Backing up to:", destination)
    subprocess.run(["rclone", "copy", str(MODEL_DIR), destination, "--checksum", "--create-empty-src-dirs"], check=True)
    subprocess.run(["rclone", "check", str(MODEL_DIR), destination, "--one-way"], check=True)
print("Both Drive copies verified for run:", RUN_ID)


Backing up to: radha:ir/NB7_TPUv5e4/llama/run_20261123_tpuv5e4_llama_01


2026/11/23 18:18:23 NOTICE: radha: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id
2026/11/23 18:24:47 NOTICE: radha: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id
2026/11/23 18:24:49 NOTICE: Google drive root 'ir/NB7_TPUv5e4/llama/run_20261123_tpuv5e4_llama_01': 0 differences found
2026/11/23 18:24:49 NOTICE: Google drive root 'ir/NB7_TPUv5e4/llama/run_20261123_tpuv5e4_llama_01': 253 matching files
2026/11/23 18:24:49 NOTICE: radha: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Backing up to: radha:ir_final/NB7_TPUv5e4/llama/run_20261123_tpuv5e4_llama_01


2026/11/23 18:33:34 NOTICE: radha: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id


Both Drive copies verified for run: run_20261123_tpuv5e4_llama_01


2026/11/23 18:33:35 NOTICE: Google drive root 'ir_final/NB7_TPUv5e4/llama/run_20261123_tpuv5e4_llama_01': 0 differences found
2026/11/23 18:33:35 NOTICE: Google drive root 'ir_final/NB7_TPUv5e4/llama/run_20261123_tpuv5e4_llama_01': 253 matching files


## 9. Final extraction audit

In [22]:
run_distributed("audit")
manifest_path = MODEL_DIR / "extraction_manifest.json"
manifest = json.loads(manifest_path.read_text())
assert manifest["status"] == "completed"
assert manifest["final_audit"]["records_recovered"] == EXPECTED_RECORDS
print(json.dumps(manifest["final_audit"], indent=2))
print("Manifest:", manifest_path)
print("Audit:", MODEL_DIR / "shard_audit.csv")

Launching: /home/amars/adnn-tpu311/bin/python /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py --nproc_per_node=4 /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_worker.py --model-key llama --run-id run_20261123_tpuv5e4_llama_01 --mode audit --dataset /home/amars/adnn/input/NB4_v2/integrated_dataset_v2.jsonl --output /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01 --expected-sha 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd --shard-size 8 --shard-start 0 --shard-end 0


I0000 00:00:1795458892.447567  131306 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/amars/adnn-tpu311/lib/python3.11/site-packages/libtpu/libtpu.so
I0000 00:00:1795458893.617481  131306 device_rt.cc:178] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1795458893.618817  131306 libtpu_version.cc:63] Setting libtpu version to: 0.0.53.1
/home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py:4: UserWarning: //TorchTpu/ApiStage/ExperimentalApi// set_tpu_launch_env is experimental and subject to change: torch.tpu.distributed.set_tpu_launch_env is experimental and subject to change.
  torch.tpu.distributed.set_tpu_launch_env()
W1123 18:34:53.658000 131306 torch/distributed/run.py:982] 
W1123 18:34:53.658000 131306 torch/distributed/run.py:982] *****************************************
W1

FINAL_AUDIT_PASS {"shards_checked": 251, "shards_passed": 251, "records_recovered": 2001, "ordered_record_ids_match": true, "dataset_sha256": "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd", "completed_utc": "2026-11-23T18:35:37.857538+00:00"}
WORKER_COMPLETE audit llama


I0000 00:00:1795458938.873986  131374 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795458938.874007  131374 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=283
num_cache_hits=266 {93.9%}
num_compilations=1
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795458938.874042  131377 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795458938.874066  131377 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=281
num_cache_hits=265 {94.3%}
num_compilations=2
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795458938.875066  131376 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795458938.875093  131376 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=284
num_cache_hits=264 {92.9%}
num_compilations=1
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795458938.875398  131375 compilat

{
  "shards_checked": 251,
  "shards_passed": 251,
  "records_recovered": 2001,
  "ordered_record_ids_match": true,
  "dataset_sha256": "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd",
  "completed_utc": "2026-11-23T18:35:37.857538+00:00"
}
Manifest: /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01/extraction_manifest.json
Audit: /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01/shard_audit.csv


## 10. Backup/package and next model

Each run is isolated by model and `RUN_ID`. Preserve the complete run directory as one unit, including its manifest, pilot (if retained), shards, and audit. Do not flatten shards from different runs into one folder.

After each successful shard batch, run the backup cell (Section 8A) to sync and verify this run directory to BOTH Drive roots, preserving the same relative path:
- `radha:ir/NB7_TPU_v5e4/<model>/<RUN_ID>/`
- `radha:ir_final/NB7_TPU_v5e4/<model>/<RUN_ID>/`

Use the same run ID for both copies. Verify remote file counts and SHA-256 checksums against the local manifest before considering a backup complete. Do not mark an incomplete or failed run as a completed extraction. After final audit, create an archive named with the model and run ID. Restart the Jupyter kernel before selecting the other model; Qwen and Llama have separate directories and manifests.


In [23]:
import hashlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

# 1. Create a uniquely named archive.
RUN_TAG = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")

archive_path = (
    OUTPUT_ROOT /
    f"NB7_{MODEL_KEY}_TPU_v5e4_{RUN_ID}_{RUN_TAG}.zip"
)

print("Creating archive:", archive_path)

with zipfile.ZipFile(
    archive_path, "w", compression=zipfile.ZIP_STORED
) as zf:
    for p in sorted(MODEL_DIR.rglob("*")):
        if p.is_file():
            zf.write(
                p,
                arcname=str(
                    Path(MODEL_DIR.name) / p.relative_to(MODEL_DIR)
                )
            )

assert archive_path.exists() and archive_path.stat().st_size > 0

print("Archive created:", archive_path)
print("Archive size (GiB):", round(archive_path.stat().st_size / 1024**3, 3))


# 2. Calculate the local archive MD5.
local_md5_hash = hashlib.md5()

with open(archive_path, "rb") as f:
    for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
        local_md5_hash.update(chunk)

local_md5 = local_md5_hash.hexdigest()

print("Local archive MD5:", local_md5)


# 3. Upload and verify the archive in both Drive roots.
REMOTE_BASE = f"NB7_TPUv5e4/{MODEL_KEY}/{RUN_ID}"

for drive_root in ("ir", "ir_final"):

    remote_archive = (
        f"radha:{drive_root}/{REMOTE_BASE}/{archive_path.name}"
    )

    print(f"\nUploading to {drive_root}...")

    subprocess.run(
        [
            "rclone", "copyto",
            str(archive_path),
            remote_archive,
            "--checksum"
        ],
        check=True
    )

    print(f"Verifying {drive_root}...")

    result = subprocess.run(
        ["rclone", "md5sum", remote_archive],
        check=True,
        capture_output=True,
        text=True
    )

    remote_md5 = result.stdout.split()[0].strip()

    assert remote_md5 == local_md5, (
        f"MD5 MISMATCH in {drive_root}: "
        f"local={local_md5}, remote={remote_md5}"
    )

    print("VERIFIED:", remote_archive)
    print("MD5:", remote_md5)


# 4. Final confirmation.
print("\n" + "=" * 65)
print("FINAL ARCHIVE VERIFIED IN BOTH DRIVE ROOTS")
print("=" * 65)
print("Run ID:", RUN_ID)
print("Archive:", archive_path.name)
print("Size (GiB):", round(archive_path.stat().st_size / 1024**3, 3))
print("MD5:", local_md5)
print("Drive roots: ir, ir_final")
print("Status: PASS")

Creating archive: /home/amars/adnn/outputs/NB7_TPUv5e4/NB7_llama_TPU_v5e4_run_20261123_tpuv5e4_llama_01_20261123T183548Z.zip
Archive created: /home/amars/adnn/outputs/NB7_TPUv5e4/NB7_llama_TPU_v5e4_run_20261123_tpuv5e4_llama_01_20261123T183548Z.zip
Archive size (GiB): 1.351
Local archive MD5: 6236cfae34facf14cf0b2bb9c9b4c3e5

Uploading to ir...


2026/11/23 18:35:54 NOTICE: radha: This remote uses rclone's shared Google Drive client_id, which is being retired and will stop working during 2026. Create your own client_id to avoid interruption: https://rclone.org/drive/#making-your-own-client-id
2026/11/23 19:00:23 ERROR : NB7_llama_TPU_v5e4_run_20261123_tpuv5e4_llama_01_20261123T183548Z.zip: Failed to copy: googleapi: Error 403: Quota exceeded for quota metric 'Queries' and limit 'Previous quota: Requests per minute' of service 'drive.googleapis.com' for consumer 'project_number:202264815644'.
Details:
[
  {
    "@type": "type.googleapis.com/google.rpc.ErrorInfo",
    "domain": "googleapis.com",
    "metadata": {
      "consumer": "projects/202264815644",
      "quota_limit": "defaultPerMinutePerProject",
      "quota_limit_value": "840000",
      "quota_location": "global",
      "quota_metric": "drive.googleapis.com/default",
      "quota_unit": "1/min/{project}",
      "service": "drive.googleapis.com"
    },
    "reason": "RA

KeyboardInterrupt: 

In [24]:
run_distributed("audit")
manifest_path = MODEL_DIR / "extraction_manifest.json"
manifest = json.loads(manifest_path.read_text())
assert manifest["status"] == "completed"
assert manifest["final_audit"]["records_recovered"] == EXPECTED_RECORDS
print(json.dumps(manifest["final_audit"], indent=2))
print("Manifest:", manifest_path)
print("Audit:", MODEL_DIR / "shard_audit.csv")

Launching: /home/amars/adnn-tpu311/bin/python /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py --nproc_per_node=4 /home/amars/adnn/scripts/NB7_TPUv5e4/nb7_worker.py --model-key llama --run-id run_20261123_tpuv5e4_llama_01 --mode audit --dataset /home/amars/adnn/input/NB4_v2/integrated_dataset_v2.jsonl --output /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01 --expected-sha 0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd --shard-size 8 --shard-start 0 --shard-end 0


I0000 00:00:1795460621.390499  133663 pjrt_api.cc:119] GetPjrtApi was found for TPU at /home/amars/adnn-tpu311/lib/python3.11/site-packages/libtpu/libtpu.so
I0000 00:00:1795460622.583456  133663 device_rt.cc:178] PjRt runtime initialization deferred for tpu
Successfully renamed PrivateUse1 backend to 'tpu'. Device: device(type='tpu')
Registered Python module for 'tpu'.
Device type: tpu, Device index: default
Initializing TPU distributed runtime
I0000 00:00:1795460622.584773  133663 libtpu_version.cc:63] Setting libtpu version to: 0.0.53.1
/home/amars/adnn/scripts/NB7_TPUv5e4/nb7_tpu_launch.py:4: UserWarning: //TorchTpu/ApiStage/ExperimentalApi// set_tpu_launch_env is experimental and subject to change: torch.tpu.distributed.set_tpu_launch_env is experimental and subject to change.
  torch.tpu.distributed.set_tpu_launch_env()
W1123 19:03:42.623000 133663 torch/distributed/run.py:982] 
W1123 19:03:42.623000 133663 torch/distributed/run.py:982] *****************************************
W1

FINAL_AUDIT_PASS {"shards_checked": 251, "shards_passed": 251, "records_recovered": 2001, "ordered_record_ids_match": true, "dataset_sha256": "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd", "completed_utc": "2026-11-23T19:04:34.373966+00:00"}
WORKER_COMPLETE audit llama


I0000 00:00:1795460675.390744  133734 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795460675.390770  133734 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=277
num_cache_hits=255 {92.0%}
num_compilations=1
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795460675.390977  133733 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795460675.391000  133733 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=283
num_cache_hits=263 {92.9%}
num_compilations=0
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795460675.391251  133731 compilation_cache.cc:356] Compilation cache evicted.
I0000 00:00:1795460675.391273  133731 compilation_cache.cc:211] CompilationCache final stats: num_cache_reqs=275
num_cache_hits=250 {90.9%}
num_compilations=3
num_unbounded_dynamic_hits=0
peak_compilation_memory_bytes=unknown

I0000 00:00:1795460675.391359  133732 compilat

{
  "shards_checked": 251,
  "shards_passed": 251,
  "records_recovered": 2001,
  "ordered_record_ids_match": true,
  "dataset_sha256": "0d890a39de3953e1a4d8f03d0cde7fad9a27ebba143e07c0b31482d8bb13c9bd",
  "completed_utc": "2026-11-23T19:04:34.373966+00:00"
}
Manifest: /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01/extraction_manifest.json
Audit: /home/amars/adnn/outputs/NB7_TPUv5e4/llama31_8b/run_20261123_tpuv5e4_llama_01/shard_audit.csv


## 11. Interpretation and limitations

- `last_action_token` is the final token of the candidate action span; `mean_action_tokens` averages action-token hidden states.
- Layer 0 is the embedding output before transformer contextualization. Layer-0 mean pooling is permutation-invariant and can collapse token-order variants; do not interpret it as contextual evidence.
- Model weights are FP16; saved representations are FP32.
- Each of four TP workers sees its local accelerator as `tpu:0`; this is expected.
- The pilot validates one record and the output contract, not scientific validity or full-run throughput.
- Extraction success is not evidence that representations encode reversibility. NB8 tests that with grouped splits, baselines, controls, and training-only model/layer selection.
- This notebook uses exactly the NB4 v2 2,001-row dataset. Legacy 2,052-row representations are not reused.